# Aula 8 — Quando NÃO usar Spark
### Pandas, Pandas API on Spark, Polars e DuckDB lado a lado

Passamos quatro aulas aprendendo Spark. Agora a pergunta honesta: **quando ele é a ferramenta errada?**

Na aula 4 vimos que, com 5 milhões de linhas em uma máquina, o Pandas empatava ou ganhava do Spark. Nos últimos anos surgiram ferramentas que exploram muito melhor uma única máquina, e hoje boa parte dos problemas "de big data" cabe nelas.

**Objetivos**
- Entender o que torna Polars e DuckDB rápidos (colunar, vetorizado, paralelo, lazy)
- Comparar as cinco ferramentas na mesma consulta: tempo, memória e sintaxe
- Usar Pandas API on Spark para migrar código Pandas
- Trocar dados entre ferramentas via Apache Arrow
- Ter um critério de decisão claro

---
## 1. Teoria

### As ferramentas

| | Pandas | Polars | DuckDB | Pandas API on Spark | Spark |
|-|--------|--------|--------|---------------------|-------|
| Interface | Python (DataFrame) | Python (expressões) | **SQL** (+ API Python) | Sintaxe do Pandas | DataFrame + SQL |
| Motor | NumPy/C | Rust | C++ | Spark | Spark (JVM) |
| Execução | Eager | Eager **ou lazy** | Lazy (otimizador SQL) | Lazy | Lazy |
| Núcleos usados | **1** (na maioria das operações) | Todos | Todos | Todos / cluster | Todos / cluster |
| Dados maiores que a RAM | Não | Sim (engine streaming) | Sim (spill em disco) | Sim | Sim |
| Várias máquinas | Não | Não | Não | **Sim** | **Sim** |
| Instalação | `pip` | `pip` | `pip` | `pip` + Java | `pip` + Java |

### Por que Polars e DuckDB são rápidos

| Técnica | Ideia | Pandas | Polars / DuckDB |
|---------|-------|--------|-----------------|
| **Armazenamento colunar** (Apache Arrow) | Valores da mesma coluna contíguos na memória | Parcial (NumPy) | Sim |
| **Execução vetorizada** | Processa lotes de milhares de valores por instrução | Parcial | Sim |
| **Paralelismo** | Divide o trabalho entre todos os núcleos | Não | Sim |
| **Otimização de consulta** | Reordena, empurra filtros, lê só as colunas usadas | Não (eager) | Sim (lazy) |
| **Out-of-core** | Processa em pedaços, usando disco quando a RAM acaba | Não | Sim |

Repare que as três últimas linhas são exatamente o que aprendemos sobre o Spark: paralelismo, avaliação preguiçosa com otimizador, predicate e column pruning. **Polars e DuckDB aplicam as mesmas ideias, sem o custo de coordenar um cluster.**

### O custo fixo do Spark

```
  Pandas / Polars / DuckDB              Spark
  ┌────────────────────────┐            ┌──────────────────────────────────────────┐
  │ processo Python        │            │ Python ─py4j─▶ JVM (driver)               │
  │ └─ biblioteca nativa   │            │                 ├─ planejamento (Catalyst) │
  │    lê, calcula, devolve│            │                 ├─ jobs, stages, tasks     │
  └────────────────────────┘            │                 ├─ shuffle (disco)         │
                                        │                 └─ resultado ─▶ Python     │
                                        └──────────────────────────────────────────┘
  custo fixo ~ milissegundos             custo fixo ~ centenas de ms a segundos
```

Esse custo é irrelevante com 1 TB em 50 máquinas. Com 2 GB em um notebook, ele domina.

---
## 2. Setup

> No Colab, Polars e DuckDB já vêm instalados. A Pandas API on Spark do PySpark 4.0 exige **pandas 2.x**: com pandas 3 ela não importa.

In [ ]:
# !pip install -q pyspark==4.0.1 "pandas>=2.2,<3" polars duckdb pyarrow

In [ ]:
import os, time, gc, subprocess, sys, warnings
os.environ["PYARROW_IGNORE_TIMEZONE"] = "1"      # exigido pela Pandas API on Spark
warnings.filterwarnings("ignore", message=".*(index_col|to_pandas|loads all data).*")   # avisos da seção 6

import numpy as np
import pandas as pd
import polars as pl
import duckdb
import pyarrow as pa
import matplotlib.pyplot as plt

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark = (
    SparkSession.builder
    .appName("Aula 8 - Quando nao usar Spark")
    .master("local[*]")
    .config("spark.driver.memory", "4g")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.sql.ansi.enabled", "false")      # a Pandas API on Spark não funciona com ANSI (ver seção 6)
    .config("spark.sql.execution.arrow.pyspark.enabled", "true")   # Arrow na troca Python <-> JVM (ver seção 9)
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")

import pyspark.pandas as ps
ps.set_option("compute.default_index_type", "distributed")   # explicado na seção 6

for name, mod in [("pandas", pd), ("polars", pl), ("duckdb", duckdb), ("pyarrow", pa)]:
    print(f"{name:<8}: {mod.__version__}")
print(f"pyspark : {spark.version}")
print(f"núcleos : {os.cpu_count()}")

---
## 3. Dados

Mesmo modelo da aula 4: pedidos, clientes e produtos, salvos em Parquet. Desta vez geramos com NumPy, sem Spark.

In [ ]:
BASE = "/tmp/aula8"
os.makedirs(BASE, exist_ok=True)

REGIONS    = np.array(["North", "Northeast", "Midwest", "Southeast", "South"])
CATEGORIES = np.array(["Computers", "Peripherals", "Audio", "Storage", "Cables"])
SEGMENTS   = np.array(["Consumer", "Corporate", "Small Business"])
N_CUSTOMERS, N_PRODUCTS = 200_000, 1_000

def generate_orders(n, seed=0):
    rng = np.random.default_rng(seed)
    return pd.DataFrame({
        "order_id":    np.arange(n, dtype="int64"),
        "customer_id": rng.integers(0, N_CUSTOMERS, n, dtype="int32"),
        "product_id":  rng.integers(0, N_PRODUCTS, n, dtype="int32"),
        "quantity":    rng.integers(1, 6, n, dtype="int32"),
        "price":       np.round(rng.uniform(10, 500, n), 2),
        "region":      REGIONS[rng.integers(0, len(REGIONS), n)],
        "order_date":  np.datetime64("2024-01-01") + rng.integers(0, 365, n).astype("timedelta64[D]"),
    })

rng = np.random.default_rng(1)
pd.DataFrame({
    "customer_id": np.arange(N_CUSTOMERS, dtype="int32"),
    "segment":     SEGMENTS[rng.integers(0, 3, N_CUSTOMERS)],
}).to_parquet(f"{BASE}/customers.parquet", index=False)

pd.DataFrame({
    "product_id":   np.arange(N_PRODUCTS, dtype="int32"),
    "product_name": [f"Product {i}" for i in range(N_PRODUCTS)],
    "category":     CATEGORIES[rng.integers(0, 5, N_PRODUCTS)],
}).to_parquet(f"{BASE}/products.parquet", index=False)

N = 5_000_000
generate_orders(N).to_parquet(f"{BASE}/orders.parquet", index=False)

ORDERS, CUSTOMERS, PRODUCTS = (f"{BASE}/{t}.parquet" for t in ["orders", "customers", "products"])
print(f"orders.parquet: {os.path.getsize(ORDERS) / 1e6:.0f} MB em disco, {N:,} linhas")

---
## 4. A mesma consulta em cinco ferramentas

**Consulta:** receita (`price * quantity`) por segmento de cliente e categoria de produto, só para pedidos do 2º semestre. Ela tem leitura, filtro, dois joins e uma agregação: o pipeline típico de um relatório.

Cada função **lê os arquivos do zero** e devolve um `pandas.DataFrame` ordenado, para podermos medir o tempo e conferir que todas chegam ao mesmo resultado.

In [ ]:
CUTOFF = "2024-07-01"

def q_pandas():
    o = pd.read_parquet(ORDERS, columns=["customer_id", "product_id", "quantity", "price", "order_date"])
    c = pd.read_parquet(CUSTOMERS)
    p = pd.read_parquet(PRODUCTS, columns=["product_id", "category"])
    o = o[o["order_date"] >= CUTOFF]
    o = o.assign(revenue=o["price"] * o["quantity"])
    df = o.merge(c, on="customer_id").merge(p, on="product_id")
    return (df.groupby(["segment", "category"], as_index=False)
              .agg(orders=("revenue", "size"), revenue=("revenue", "sum")))

def q_polars():
    o = pl.scan_parquet(ORDERS)
    c = pl.scan_parquet(CUSTOMERS)
    p = pl.scan_parquet(PRODUCTS)
    return (
        o.filter(pl.col("order_date") >= pl.date(2024, 7, 1))
         .with_columns(revenue=pl.col("price") * pl.col("quantity"))
         .join(c, on="customer_id")
         .join(p, on="product_id")
         .group_by("segment", "category")
         .agg(orders=pl.len(), revenue=pl.col("revenue").sum())
         .collect()
         .to_pandas()
    )

def q_duckdb():
    return duckdb.sql(f"""
        SELECT c.segment, p.category,
               COUNT(*)                   AS orders,
               SUM(o.price * o.quantity)  AS revenue
        FROM '{ORDERS}'    o
        JOIN '{CUSTOMERS}' c USING (customer_id)
        JOIN '{PRODUCTS}'  p USING (product_id)
        WHERE o.order_date >= DATE '{CUTOFF}'
        GROUP BY ALL
    """).df()

def q_spark():
    o = spark.read.parquet(ORDERS)
    c = spark.read.parquet(CUSTOMERS)
    p = spark.read.parquet(PRODUCTS)
    return (
        o.filter(F.col("order_date") >= CUTOFF)
         .withColumn("revenue", F.col("price") * F.col("quantity"))
         .join(c, "customer_id")
         .join(p, "product_id")
         .groupBy("segment", "category")
         .agg(F.count("*").alias("orders"), F.sum("revenue").alias("revenue"))
         .toPandas()
    )

def q_pandas_on_spark():
    o = ps.read_parquet(ORDERS, columns=["customer_id", "product_id", "quantity", "price", "order_date"])
    c = ps.read_parquet(CUSTOMERS)
    p = ps.read_parquet(PRODUCTS, columns=["product_id", "category"])
    o = o[o["order_date"] >= CUTOFF]
    o["revenue"] = o["price"] * o["quantity"]
    df = o.merge(c, on="customer_id").merge(p, on="product_id")
    out = df.groupby(["segment", "category"]).agg(orders=("revenue", "count"), revenue=("revenue", "sum"))
    return out.reset_index().to_pandas()

QUERIES = {
    "Pandas": q_pandas, "Polars": q_polars, "DuckDB": q_duckdb,
    "Spark": q_spark, "Pandas API on Spark": q_pandas_on_spark,
}

Repare na sintaxe:
- **Pandas API on Spark** é praticamente o código Pandas com `pd` trocado por `ps`.
- **Polars** lembra muito o PySpark: `filter`, `with_columns`, `join`, `group_by`, `agg`, com expressões `pl.col(...)` no lugar de `F.col(...)`.
- **DuckDB** é SQL puro, lendo os arquivos Parquet diretamente no `FROM`, sem nenhum passo de leitura.

In [ ]:
def normalize(df):
    df = df[["segment", "category", "orders", "revenue"]].copy()
    df["orders"] = df["orders"].astype("int64")
    return df.sort_values(["segment", "category"]).reset_index(drop=True)

def benchmark(queries, repeat=3):
    """Roda cada consulta `repeat` vezes e guarda o melhor tempo (a 1a execução inclui aquecimento)."""
    times, results = {}, {}
    for name, fn in queries.items():
        best = float("inf")
        for _ in range(repeat):
            gc.collect()
            t0 = time.perf_counter()
            res = fn()
            best = min(best, time.perf_counter() - t0)
        times[name], results[name] = best, normalize(res)
        print(f"{name:<22} {best:6.2f} s")
    return times, results

times, results = benchmark(QUERIES)

# Todas as ferramentas chegam ao mesmo resultado?
ref = results["Pandas"]
for name, res in results.items():
    pd.testing.assert_frame_equal(ref, res, check_exact=False, rtol=1e-9)
print("\nResultados idênticos nas 5 ferramentas.")
ref.head()

In [ ]:
def bar_chart(times, title):
    names = sorted(times, key=times.get)
    fig, ax = plt.subplots(figsize=(7, 3))
    ax.barh(names, [times[n] for n in names], color="#4c78a8")
    for i, n in enumerate(names):
        ax.text(times[n], i, f" {times[n]:.2f} s", va="center")
    ax.invert_yaxis()
    ax.set_xlabel("segundos (menor é melhor)")
    ax.set_title(title)
    ax.spines[["top", "right"]].set_visible(False)
    plt.tight_layout(); plt.show()

bar_chart(times, f"Join + agregação, {N/1e6:.0f} milhões de pedidos, {os.cpu_count()} núcleos")

**Como ler o resultado** (os números variam com a máquina, mas a ordem costuma se repetir):

- **Polars e DuckDB** costumam ser os mais rápidos, muitas vezes por uma boa margem.
- O **Pandas** fica no meio: lê rápido, mas faz os joins em um único núcleo e materializa cada passo intermediário.
- O **Spark** paga o custo fixo de planejamento, tasks e transferência para o Python.
- A **Pandas API on Spark** soma ao custo do Spark a tradução da semântica do Pandas (índice, ordem). Ela não foi feita para ser rápida em uma máquina, e sim para **escalar código Pandas existente**.

### 4.1 Memória

Tempo é só metade da história: o Pandas costuma quebrar por **falta de memória** bem antes de ficar lento demais.

Para medir de forma justa, rodamos cada ferramenta em um **processo Python separado** e lemos o pico de memória do processo (`VmHWM`, o maior RSS que ele atingiu). O Spark fica de fora: a JVM reserva a memória configurada logo ao iniciar (`spark.driver.memory`), então o pico dela não diz quanto a consulta precisou.

In [ ]:
MEMORY_SCRIPTS = {
    "Pandas": f"""
import pandas as pd
o = pd.read_parquet('{ORDERS}', columns=['customer_id','product_id','quantity','price','order_date'])
c = pd.read_parquet('{CUSTOMERS}'); p = pd.read_parquet('{PRODUCTS}', columns=['product_id','category'])
o = o[o['order_date'] >= '{CUTOFF}']; o = o.assign(revenue=o['price'] * o['quantity'])
o.merge(c, on='customer_id').merge(p, on='product_id').groupby(['segment','category'])['revenue'].sum()
""",
    "Polars": f"""
import polars as pl
(pl.scan_parquet('{ORDERS}').filter(pl.col('order_date') >= pl.date(2024, 7, 1))
   .with_columns(revenue=pl.col('price') * pl.col('quantity'))
   .join(pl.scan_parquet('{CUSTOMERS}'), on='customer_id').join(pl.scan_parquet('{PRODUCTS}'), on='product_id')
   .group_by('segment', 'category').agg(pl.col('revenue').sum()).collect())
""",
    "DuckDB": f"""
import duckdb
duckdb.sql(\"\"\"SELECT segment, category, SUM(price * quantity) FROM '{ORDERS}'
  JOIN '{CUSTOMERS}' USING (customer_id) JOIN '{PRODUCTS}' USING (product_id)
  WHERE order_date >= DATE '{CUTOFF}' GROUP BY ALL\"\"\").df()
""",
}

def peak_memory_mb(script):
    # VmHWM = pico de memória residente do processo (Linux, em KB)
    wrapper = script + "\nprint([l for l in open('/proc/self/status') if l.startswith('VmHWM')][0].split()[1])"
    out = subprocess.run([sys.executable, "-c", wrapper], capture_output=True, text=True, check=True)
    return int(out.stdout.strip().splitlines()[-1]) / 1024

baseline = peak_memory_mb("import pandas, polars, duckdb")
print(f"Processo Python vazio (só imports): {baseline:,.0f} MB\n")

memory = {name: peak_memory_mb(s) for name, s in MEMORY_SCRIPTS.items()}
for name, mb in memory.items():
    print(f"{name:<8} pico de memória: {mb:7,.0f} MB")
print(f"\nArquivo orders.parquet em disco: {os.path.getsize(ORDERS) / 1e6:,.0f} MB")

O Pandas carrega as colunas inteiras e cria cópias a cada `merge`. Polars e DuckDB leem em lotes, filtram cedo e mantêm só o necessário. Regra prática antiga do Pandas: **reserve de 5 a 10 vezes o tamanho dos dados em RAM.**

---
## 5. Como o resultado muda com o volume

Vamos repetir a consulta para 1, 5 e 20 milhões de linhas, comparando as quatro ferramentas principais.

> No Colab gratuito (cerca de 12 GB de RAM), 20 milhões funciona. Em máquinas menores, troque por 10 milhões.

In [ ]:
SIZES = [1_000_000, 5_000_000, 20_000_000]
scaling = {name: [] for name in ["Pandas", "Polars", "DuckDB", "Spark"]}

ORDERS_ORIGINAL = ORDERS
for n in SIZES:
    ORDERS = f"{BASE}/orders_{n // 1_000_000}M.parquet"
    if not os.path.exists(ORDERS):
        generate_orders(n, seed=n).to_parquet(ORDERS, index=False)
    print(f"--- {n:,} linhas")
    t, _ = benchmark({k: QUERIES[k] for k in scaling}, repeat=2)
    for k in scaling:
        scaling[k].append(t[k])
ORDERS = ORDERS_ORIGINAL

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
for name, ts in scaling.items():
    ax.plot([s / 1e6 for s in SIZES], ts, marker="o", label=name)
ax.set_xlabel("milhões de linhas"); ax.set_ylabel("segundos")
ax.set_title("Tempo da consulta vs volume de dados")
ax.spines[["top", "right"]].set_visible(False)
ax.legend(frameon=False); plt.tight_layout(); plt.show()

O que observar:
- O custo fixo do Spark pesa mais no volume pequeno; conforme o volume cresce, a diferença relativa diminui.
- O Pandas cresce mais rápido que os outros, e em algum ponto deixa de caber na memória.
- Polars e DuckDB crescem devagar: usam todos os núcleos e não materializam passos intermediários.

**A curva que este notebook não consegue mostrar** é a do Spark em um cluster: com 10 máquinas, o mesmo código processa 10 vezes mais dados no mesmo tempo. Nenhuma das outras ferramentas faz isso.

---
## 6. Pandas API on Spark em detalhe

`pyspark.pandas` (antigo projeto Koalas) implementa grande parte da API do Pandas sobre o Spark. O caso de uso é **migração**: você tem um código Pandas que funciona, os dados cresceram e não cabem mais em uma máquina.

```python
import pandas as pd              import pyspark.pandas as ps
df = pd.read_parquet(path)  ->   df = ps.read_parquet(path)
df.groupby("x")["y"].mean()      df.groupby("x")["y"].mean()      # mesmo código
```

### Diferenças que importam

| Tema | Pandas | Pandas API on Spark |
|------|--------|---------------------|
| Índice | Sempre existe, ordenado | Precisa ser **gerado**; o tipo afeta muito a performance |
| Ordem das linhas | Garantida | Não garantida após operações distribuídas |
| Execução | Imediata | Lazy (vira um plano Spark) |
| `df.apply(func)` linha a linha | Lento | Muito lento (UDF Python, aula 4) |
| `iterrows`, `.values`, `.to_numpy()` | Normais | Trazem **tudo** para o driver |
| Modo ANSI do Spark 4 | — | Precisa estar desligado |

### Tipos de índice padrão (`compute.default_index_type`)

| Tipo | Como gera | Custo |
|------|-----------|-------|
| `sequence` (padrão) | 0, 1, 2... em ordem, **em uma única partição** | Gargalo: todos os dados passam por uma task |
| `distributed-sequence` | 0, 1, 2... calculado por partição | Médio (um job extra) |
| `distributed` | IDs crescentes, mas **não consecutivos** | Baixo; use quando o índice não importa |

In [ ]:
psdf = ps.read_parquet(ORDERS, columns=["order_id", "region", "price", "quantity"])
print(type(psdf))
psdf.head(3)

In [ ]:
# Por baixo, tudo é um plano Spark
summary = psdf.groupby("region")["price"].mean().sort_values(ascending=False)
summary.to_frame().spark.explain()
summary

In [ ]:
# Transição entre os mundos
sdf  = psdf.to_spark()            # pandas-on-Spark -> Spark DataFrame (sem custo)
back = sdf.pandas_api()           # Spark DataFrame -> pandas-on-Spark (gera índice)
small = psdf.head(1000).to_pandas()   # -> pandas de verdade: só para resultados PEQUENOS
print(type(sdf).__name__, "|", type(back).__name__, "|", type(small).__name__)

In [ ]:
# O custo do índice: o mesmo groupby com cada tipo de índice padrão
for idx in ["sequence", "distributed-sequence", "distributed"]:
    ps.set_option("compute.default_index_type", idx)
    t0 = time.perf_counter()
    ps.read_parquet(ORDERS, columns=["region", "price"]).groupby("region")["price"].sum().to_pandas()
    print(f"{idx:<22} {time.perf_counter() - t0:5.2f} s")
ps.set_option("compute.default_index_type", "distributed")

### Por que desligamos o ANSI mode

No Spark 4, o modo ANSI (aula 5) está ligado por padrão. A Pandas API on Spark imita comportamentos do Pandas que conflitam com ele (por exemplo, divisão por zero devolvendo `inf`/`NaN` em vez de erro). Por isso ela se recusa a iniciar com ANSI ligado. Por isso criamos a sessão com `spark.sql.ansi.enabled=false`.

---
## 7. Polars em detalhe

O Polars tem dois modos:
- **Eager** (`pl.read_parquet`, `pl.DataFrame`): executa cada passo na hora, como o Pandas.
- **Lazy** (`pl.scan_parquet`, `df.lazy()`): monta um plano e só executa no `.collect()`, como o Spark.

O plano lazy passa por um otimizador com as mesmas ideias do Catalyst.

In [ ]:
lazy = (
    pl.scan_parquet(ORDERS)
    .with_columns(revenue=pl.col("price") * pl.col("quantity"))
    .filter(pl.col("region") == "South")
    .group_by("product_id")
    .agg(pl.col("revenue").sum())
    .sort("revenue", descending=True)
    .head(5)
)
print(lazy.explain())

No plano otimizado, a leitura do Parquet já recebe o filtro (`SELECTION`) e só as colunas usadas (`PROJECT 4/7 COLUMNS`): predicate e column pruning, como no `FileScan` do Spark.

### Expressões e window functions

Equivalências com o PySpark:

| PySpark | Polars |
|---------|--------|
| `F.col("x")` | `pl.col("x")` |
| `df.withColumn("y", expr)` | `df.with_columns(y=expr)` |
| `df.groupBy("k").agg(...)` | `df.group_by("k").agg(...)` |
| `F.when(c, a).otherwise(b)` | `pl.when(c).then(a).otherwise(b)` |
| `F.sum("x").over(Window.partitionBy("k"))` | `pl.col("x").sum().over("k")` |
| `F.rank().over(...)` | `pl.col("x").rank(...).over("k")` |

In [ ]:
top3 = (
    pl.scan_parquet(ORDERS)
    .join(pl.scan_parquet(PRODUCTS), on="product_id")
    .group_by("region", "product_name")
    .agg(revenue=(pl.col("price") * pl.col("quantity")).sum())
    .with_columns(rank=pl.col("revenue").rank("dense", descending=True).over("region"))
    .filter(pl.col("rank") <= 3)
    .sort("region", "rank")
    .collect()
)
top3

In [ ]:
# Engine streaming: processa em lotes, permitindo dados maiores que a memória.
# O código é o mesmo; muda só o argumento do collect.
for engine in ["in-memory", "streaming"]:
    t0 = time.perf_counter()
    res = lazy.collect(engine=engine)
    print(f"engine {engine:<10} {time.perf_counter() - t0:.2f} s")
res

---
## 8. DuckDB em detalhe

O DuckDB é um banco de dados analítico **embutido**: não há servidor, ele roda dentro do processo Python, como o SQLite, mas otimizado para consultas analíticas.

Três recursos que o tornam muito prático:
1. **SQL direto sobre arquivos**: Parquet, CSV, JSON, inclusive com curingas (`'pasta/*.parquet'`) e em S3.
2. **SQL direto sobre DataFrames** Pandas, Polars ou Arrow que estão em variáveis Python, sem cópia.
3. **Out-of-core**: quando a memória acaba, ele usa disco em vez de quebrar.

In [ ]:
# 1) SQL sobre arquivo, com EXPLAIN
print(duckdb.sql(f"EXPLAIN SELECT region, AVG(price) FROM '{ORDERS}' WHERE quantity > 3 GROUP BY region").fetchall()[0][1])

In [ ]:
# 2) SQL sobre um DataFrame Pandas que está na memória (o nome da variável vira tabela)
regions_pd = pd.DataFrame({"region": REGIONS, "manager": ["Ana", "Bruno", "Carla", "Diego", "Elisa"]})

duckdb.sql(f"""
    SELECT r.manager, o.region, ROUND(SUM(o.price * o.quantity), 2) AS revenue
    FROM '{ORDERS}' o
    JOIN regions_pd r USING (region)
    GROUP BY ALL
    ORDER BY revenue DESC
""").pl()      # .df() -> Pandas | .pl() -> Polars | .arrow() -> Arrow | .fetchall() -> tuplas

In [ ]:
# 3) Out-of-core: limitamos a memória a 300 MB e fazemos uma agregação com 5 milhões de grupos
con = duckdb.connect()
con.sql("SET memory_limit = '300MB'")
con.sql(f"SET temp_directory = '{BASE}/duckdb_tmp'")

t0 = time.perf_counter()
n_groups = con.execute(f"""
    SELECT COUNT(*) FROM (
        SELECT order_id, SUM(price) FROM '{ORDERS}' GROUP BY order_id
    )
""").fetchone()[0]
print(f"{n_groups:,} grupos com limite de 300 MB em {time.perf_counter() - t0:.2f} s")
con.close()

In [ ]:
# DuckDB também escreve: um Parquet particionado, como o partitionBy do Spark
duckdb.sql(f"""
    COPY (SELECT *, month(order_date) AS month FROM '{ORDERS}' WHERE region = 'South')
    TO '{BASE}/south_by_month' (FORMAT parquet, PARTITION_BY (month), OVERWRITE_OR_IGNORE)
""")
print(sorted(os.listdir(f"{BASE}/south_by_month"))[:4], "...")

---
## 9. Apache Arrow: a língua comum

Todas essas ferramentas falam **Apache Arrow**, um formato colunar padrão na memória. Por isso trocar dados entre elas é rápido, muitas vezes sem cópia.

```
                        ┌───────────────┐
        Pandas ◀──────▶ │               │ ◀──────▶ Polars
                        │ Apache Arrow  │
        DuckDB ◀──────▶ │ (colunar, na  │ ◀──────▶ Spark (toArrow / createDataFrame)
                        │  memória)     │
       Parquet ◀──────▶ │               │ ◀──────▶ ML (PyTorch, scikit-learn via NumPy)
                        └───────────────┘
```

Isso permite **combinar ferramentas**: Spark para o ETL pesado no cluster, DuckDB para explorar o resultado, Polars para preparar features e Pandas para o gráfico final.

In [ ]:
arrow_tbl = duckdb.sql(f"SELECT * FROM '{ORDERS}' LIMIT 1000000").arrow()
if not isinstance(arrow_tbl, pa.Table):          # versões recentes do DuckDB devolvem um RecordBatchReader
    arrow_tbl = arrow_tbl.read_all()

pdf_1m = arrow_tbl.to_pandas()

conversions = {
    "Arrow  -> Polars": lambda: pl.from_arrow(arrow_tbl),
    "Arrow  -> Pandas": lambda: arrow_tbl.to_pandas(),
    "Pandas -> DuckDB": lambda: duckdb.sql("SELECT COUNT(*) FROM pdf_1m").fetchall(),
    "Polars -> Pandas": lambda: pl.from_arrow(arrow_tbl).to_pandas(),
    "Pandas -> Spark":  lambda: spark.createDataFrame(pdf_1m).count(),
    "Spark  -> Pandas": lambda: spark.read.parquet(ORDERS).limit(1_000_000).toPandas(),
}
print("1 milhão de linhas:")
for name, fn in conversions.items():
    t0 = time.perf_counter(); fn()
    print(f"  {name:<18} {time.perf_counter() - t0:6.3f} s")

Conversões entre Pandas, Polars, DuckDB e Arrow levam milissegundos. Entrar e sair do Spark custa mais, porque os dados atravessam do Python para a JVM.

### Arrow no Spark: uma configuração que faz muita diferença

No PySpark 4.0, o uso de Arrow em `toPandas()` e `createDataFrame(pandas_df)` ainda vem **desligado por padrão** (`spark.sql.execution.arrow.pyspark.enabled=false`). Nós o ligamos no setup. Veja a diferença:

In [ ]:
sample = pdf_1m.head(300_000)
for enabled in ["false", "true"]:
    spark.conf.set("spark.sql.execution.arrow.pyspark.enabled", enabled)
    t0 = time.perf_counter(); spark.createDataFrame(sample).count(); t_in = time.perf_counter() - t0
    t0 = time.perf_counter(); spark.read.parquet(ORDERS).limit(300_000).toPandas(); t_out = time.perf_counter() - t0
    print(f"Arrow {enabled:<5} | Pandas -> Spark {t_in:6.2f} s | Spark -> Pandas {t_out:6.2f} s")
spark.conf.set("spark.sql.execution.arrow.pyspark.enabled", "true")

Sem Arrow, cada linha é convertida em objeto Python e serializada individualmente. Com Arrow, colunas inteiras são transferidas em lotes. **Ligue sempre essa configuração quando trocar dados entre Pandas e Spark.**

---
## 10. Como decidir

```
                      Os dados cabem confortavelmente
                      em UMA máquina (inclusive com spill em disco)?
                              │
               ┌──── NÃO ─────┴───── SIM ────┐
               ▼                             ▼
            SPARK                   O que já existe / o time prefere?
     (cluster, streaming,                    │
      Delta, MLlib distribuído)   ┌──────────┼────────────┬─────────────────┐
               │                  ▼          ▼            ▼                 ▼
     Código Pandas legado    Time de SQL   Pipeline em   Exploração        Código Pandas
     que precisa escalar?        │         Python novo   rápida, < 1 GB    legado pequeno
               │                 ▼             ▼            ▼                 ▼
     PANDAS API ON SPARK      DUCKDB        POLARS       PANDAS            PANDAS
```

| Situação | Ferramenta |
|----------|------------|
| Dezenas de GB a TB, ou dados que vão crescer muito | Spark |
| Streaming, Delta Lake, MLlib, já existe um cluster | Spark |
| Código Pandas que não cabe mais na memória | Pandas API on Spark (ou reescrever em Polars) |
| Até dezenas de GB em uma máquina, time que pensa em SQL | DuckDB |
| Até dezenas de GB em uma máquina, pipeline em Python | Polars |
| Exploração rápida, poucos MB, integração com tudo | Pandas |

Dois avisos para não exagerar no sentido contrário:
- "Cabe em uma máquina" depende da máquina. Uma VM na nuvem com 256 GB de RAM processa muita coisa com DuckDB ou Polars, e às vezes custa menos que um cluster.
- Spark não é só velocidade: ele traz **tolerância a falhas, escala horizontal e um ecossistema** (Delta, streaming, catálogos, agendamento) que as outras ferramentas não substituem.

---
## 11. Exercícios

In [ ]:
# ── Exercício 1 ─────────────────────────────────────────────────────────────
# Ticket médio (receita / pedidos) por mês, só para a região "Southeast".
# Escreva em Polars (lazy) e em DuckDB, confira que os resultados batem e compare os tempos.

# Sua solução:

In [ ]:
# ── Exercício 2 ─────────────────────────────────────────────────────────────
# O código Pandas abaixo calcula, para cada cliente, a receita total e o número de pedidos,
# e depois filtra os 100 clientes de maior receita. Migre-o para a Pandas API on Spark
# mudando o mínimo possível e confira que o resultado é o mesmo.

def top_customers_pandas():
    o = pd.read_parquet(ORDERS, columns=["customer_id", "price", "quantity"])
    o["revenue"] = o["price"] * o["quantity"]
    agg = o.groupby("customer_id").agg(revenue=("revenue", "sum"), orders=("revenue", "count"))
    return agg.sort_values("revenue", ascending=False).head(100)

# Sua solução:

In [ ]:
# ── Exercício 3 ─────────────────────────────────────────────────────────────
# Use o DuckDB para consultar um DataFrame POLARS que está em memória:
# crie em Polars uma tabela de metas mensais por região (meta = 65.000.000 para todas),
# e com SQL compare a receita real de cada mês/região com a meta (diferença e % atingido).

# Sua solução:

**Exercício 4 (discussão).** Qual ferramenta você escolheria em cada cenário? Justifique.

1. Um analista recebe um CSV de 300 MB por e-mail e precisa de três gráficos até o fim do dia.
2. Um pipeline diário lê 2 TB de logs no S3, junta com tabelas Delta e grava a camada Silver.
3. Um time de BI quer consultar arquivos Parquet de 15 GB com SQL, no próprio notebook, sem cluster.
4. Um modelo de previsão em Pandas funcionava com 1 ano de histórico; agora precisa rodar com 10 anos (80 GB).
5. Uma API precisa responder em menos de 100 ms com agregações sobre um arquivo de 500 MB.

### Soluções

In [ ]:
# ── Solução 1 ───────────────────────────────────────────────────────────────
t0 = time.perf_counter()
sol_pl = (
    pl.scan_parquet(ORDERS)
    .filter(pl.col("region") == "Southeast")
    .group_by(month=pl.col("order_date").dt.month())
    .agg(avg_ticket=(pl.col("price") * pl.col("quantity")).mean())
    .sort("month")
    .collect()
)
t_pl = time.perf_counter() - t0

t0 = time.perf_counter()
sol_db = duckdb.sql(f"""
    SELECT month(order_date) AS month, AVG(price * quantity) AS avg_ticket
    FROM '{ORDERS}' WHERE region = 'Southeast'
    GROUP BY month ORDER BY month
""").pl()
t_db = time.perf_counter() - t0

print(f"Polars {t_pl:.2f} s | DuckDB {t_db:.2f} s")
print("Iguais:", np.allclose(sol_pl["avg_ticket"].to_numpy(), sol_db["avg_ticket"].to_numpy()))
sol_pl.head(3)

In [ ]:
# ── Solução 2 ───────────────────────────────────────────────────────────────
def top_customers_ps():
    o = ps.read_parquet(ORDERS, columns=["customer_id", "price", "quantity"])   # pd -> ps
    o["revenue"] = o["price"] * o["quantity"]
    agg = o.groupby("customer_id").agg(revenue=("revenue", "sum"), orders=("revenue", "count"))
    return agg.sort_values("revenue", ascending=False).head(100).to_pandas()    # + to_pandas()

a, b = top_customers_pandas(), top_customers_ps()
pd.testing.assert_frame_equal(a.sort_index(), b.sort_index(), check_dtype=False)
print("Resultados iguais. Mudanças: pd -> ps e .to_pandas() no resultado final (pequeno).")

In [ ]:
# ── Solução 3 ───────────────────────────────────────────────────────────────
targets = pl.DataFrame({"region": REGIONS.tolist()}).join(
    pl.DataFrame({"month": list(range(1, 13))}), how="cross"
).with_columns(target=pl.lit(65_000_000))

duckdb.sql(f"""
    WITH real AS (
        SELECT region, month(order_date) AS month, SUM(price * quantity) AS revenue
        FROM '{ORDERS}' GROUP BY ALL
    )
    SELECT t.region, t.month,
           ROUND(r.revenue)                        AS revenue,
           ROUND(r.revenue - t.target)             AS diff,
           ROUND(100 * r.revenue / t.target, 1)    AS pct_target
    FROM targets t JOIN real r USING (region, month)
    ORDER BY pct_target DESC
    LIMIT 5
""").pl()

**Solução 4**

1. **Pandas** (ou Polars). 300 MB cabem com folga; o custo de montar qualquer outra coisa não se paga.
2. **Spark.** 2 TB, fonte distribuída e tabelas Delta: é exatamente o caso de uso dele.
3. **DuckDB.** SQL direto nos arquivos, sem servidor, com spill em disco se faltar memória.
4. **Pandas API on Spark** para migrar rápido reaproveitando o código; no médio prazo, avaliar reescrever em Polars (se uma máquina grande bastar) ou em PySpark.
5. **DuckDB ou Polars** carregados uma vez na memória do serviço. O Spark não serve para latência de milissegundos por consulta.

---
## Resumo

| | Pandas | Polars | DuckDB | Pandas API on Spark | Spark |
|-|--------|--------|--------|---------------------|-------|
| Melhor para | Exploração, dados pequenos | Pipelines Python em 1 máquina | SQL analítico em 1 máquina | Escalar código Pandas | Dados distribuídos, ecossistema |
| Ler Parquet | `pd.read_parquet` | `pl.scan_parquet` | `FROM 'arquivo.parquet'` | `ps.read_parquet` | `spark.read.parquet` |
| Agregar | `groupby().agg()` | `group_by().agg()` | `GROUP BY` | `groupby().agg()` | `groupBy().agg()` |
| Lazy | Não | `scan_*` / `.lazy()` | Sempre | Sim | Sim |
| Ver o plano | — | `.explain()` | `EXPLAIN` | `.spark.explain()` | `.explain()` |
| Para Pandas | — | `.to_pandas()` | `.df()` | `.to_pandas()` | `.toPandas()` |

**Próxima aula:** Hadoop, a base sobre a qual o Spark foi construído: HDFS, MapReduce, YARN e como tudo isso se conecta com o que vimos.

In [ ]:
spark.stop()